In [ ]:
import pandas as pd
import numpy as np
import os


1. Load dataset

In [11]:
from google.colab import files

uploaded = files.upload()


Saving retail_sales_dataset.csv to retail_sales_dataset.csv


In [12]:
import pandas as pd

df_raw = pd.read_csv("retail_sales_dataset.csv")
df_raw.head()


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [13]:
df_raw.info()
df_raw.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB


,0
Transaction ID,0
Date,0
Customer ID,0
Gender,0
Age,0
Product Category,0
Quantity,0
Price per Unit,0
Total Amount,0


2. Create folders → raw / processed / output

In [14]:
base_dir = "/content/etl_project"

folders = ["raw", "processed", "output"]

for folder in folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

base_dir


'/content/etl_project'

In [16]:
df_raw.to_csv(
    os.path.join(base_dir, "raw", "retail_sales_raw.csv"),
    index=False
)


3. Clean missing values + duplicates

In [18]:
df = df_raw.copy()


In [19]:
# Numeric columns → median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical columns → mode
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [20]:
rows_before = df.shape[0]
df = df.drop_duplicates()
rows_after = df.shape[0]

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)


Rows before cleaning: 1000
Rows after cleaning: 1000


4. Standardize column names and datatypes

In [21]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df.columns


Index(['transaction_id', 'date', 'customer_id', 'gender', 'age',
       'product_category', 'quantity', 'price_per_unit', 'total_amount'],
      dtype='object')

In [22]:
for col in df.columns:
    if "date" in col:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df.dtypes


,0
transaction_id,int64
date,datetime64[ns]
customer_id,object
gender,object
age,int64
product_category,object
quantity,int64
price_per_unit,int64
total_amount,int64


In [23]:
df.to_csv(
    os.path.join(base_dir, "processed", "retail_sales_cleaned.csv"),
    index=False
)


5. Create derived columns (margin, segment flags)

In [26]:
df.columns


Index(['transaction_id', 'date', 'customer_id', 'gender', 'age',
       'product_category', 'quantity', 'price_per_unit', 'total_amount'],
      dtype='object')

In [27]:
list(df.columns)


['transaction_id',
 'date',
 'customer_id',
 'gender',
 'age',
 'product_category',
 'quantity',
 'price_per_unit',
 'total_amount']

In [30]:
df.rename(columns={"total_amount": "sales"}, inplace=True)


In [31]:
df.columns


Index(['transaction_id', 'date', 'customer_id', 'gender', 'age',
       'product_category', 'quantity', 'price_per_unit', 'sales'],
      dtype='object')

In [32]:
df["avg_item_value"] = df["sales"] / df["quantity"]


In [33]:
df["is_high_value_transaction"] = np.where(df["sales"] > 1000, 1, 0)


In [34]:
df["is_bulk_purchase"] = np.where(df["quantity"] >= 5, 1, 0)


In [35]:
if "profit" in df.columns:
    df["margin"] = df["profit"] / df["sales"]
else:
    print("Profit data not available — margin not calculated")


Profit data not available — margin not calculated


6. Splitting tables

In [36]:
customers = df[
    ["customer_id", "gender", "age"]
].drop_duplicates()


In [37]:
products = df[
    ["product_category", "price_per_unit"]
].drop_duplicates()

In [38]:
transactions = df.drop(columns=["gender", "age"])


In [39]:
print("Raw rows:", df_raw.shape[0])
print("Cleaned rows:", df.shape[0])
print("Customers:", customers.shape[0])
print("Products:", products.shape[0])
print("Transactions:", transactions.shape[0])


Raw rows: 1000
Cleaned rows: 1000
Customers: 1000
Products: 15
Transactions: 1000


8. Validate counts before & after ETL

In [43]:
print("Raw dataset rows:", df_raw.shape[0])
print("Cleaned dataset rows:", df.shape[0])

print("Unique product categories:", df["product_category"].nunique())
print("Rows in products table:", products.shape[0])

print("Transactions table rows:", transactions.shape[0])
print("Cleaned dataset rows:", df.shape[0])

assert customers.shape[0] <= df.shape[0]
assert products.shape[0] <= df.shape[0]
assert transactions.shape[0] == df.shape[0]

print("Validation checks passed successfully")


Raw dataset rows: 1000
Cleaned dataset rows: 1000
Unique product categories: 3
Rows in products table: 15
Transactions table rows: 1000
Cleaned dataset rows: 1000
Validation checks passed successfully


In [44]:
from google.colab import files

files.download("/content/etl_project/output/customers.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
import os

base_path = "/content/etl_project"

folders = ["raw", "processed", "output"]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)


In [48]:
customers.to_csv("/content/etl_project/output/customers.csv", index=False)
products.to_csv("/content/etl_project/output/products.csv", index=False)
transactions.to_csv("/content/etl_project/output/transactions.csv", index=False)


In [49]:
os.listdir("/content/etl_project/output")


['transactions.csv', 'customers.csv', 'products.csv']

In [50]:
from google.colab import files

files.download("/content/etl_project/output/customers.csv")
files.download("/content/etl_project/output/products.csv")
files.download("/content/etl_project/output/transactions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>